# Step-Back Prompting

**Step-Back Prompting** is a **Query Translation** technique in Retrieval-Augmented Generation (RAG). Instead of directly retrieving documents using the user's original question, the LLM first generates a broader, more general version of the question (a _step-back question_). This broader query is then used to retrieve richer background information, which helps the LLM produce a more accurate and well-informed answer to the original question.

---

## Why Step-Back Prompting?

Some user questions are too specific and may not retrieve enough relevant information from the knowledge base.

Step-Back Prompting addresses this by:

- Generating a broader version of the original question.
- Retrieving more comprehensive background context.
- Combining the retrieved context with the original question.
- Producing a more accurate and grounded final answer.

---

## Workflow

```text
            Original User Question
                     │
                     ▼
        Generate Step-Back Question
                     │
                     ▼
         Retrieve Background Context
                     │
                     ▼
Original Question + Retrieved Context
                     │
                     ▼
                    LLM
                     │
                     ▼
               Final Answer
```

---

## Example

**Original Question**

```text
Why do LLM agents use task decomposition?
```

**Step-Back Question**

```text
What is task decomposition in AI systems?
```

The retriever searches using the broader question, retrieves useful background knowledge, and the LLM uses that context to answer the original question.

---

## Advantages

- Improves retrieval quality for specific questions.
- Provides broader background knowledge.
- Helps answer reasoning and multi-hop questions.
- Reduces the chance of missing relevant documents.
- Produces more grounded and informative responses.

---


In [1]:
# Basic RAG setup run gareko
%run ./pipeline/01_basic_rag.ipynb

# Indexing run gareko (documents, embeddings, vectorstore, retriever)
%run ./pipeline/02_indexing_rag.ipynb

1.3.13



C:\Users\Acer\AppData\Local\Temp\ipykernel_30740\3494673254.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558


In [2]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# Step-Back Prompting ko lagi few-shot examples
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "What can the members of The Police do?",
    },
    {
        "input": "Jan Sindel was born in what country?",
        "output": "What is Jan Sindel's personal history?",
    },
]


# Few-shot examples ko prompt template create gareko
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)


# Few-shot prompt create gareko
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)


# Step-Back Prompting ko main prompt create gareko
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert in world knowledge.

Your task is to convert a specific user question into a broader, more general step-back question that is easier to answer.

Here are a few examples:
""",
        ),
        # Few-shot examples add gareko
        few_shot_prompt,
        # User ko original question
        ("user", "{question}"),
    ]
)

In [3]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# Local Ollama LLM use gareko (OpenAI ko satta)
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Step-Back question generate garne chain
generate_queries_step_back = prompt | llm | StrOutputParser()


# User ko original question
question = "What is task decomposition for LLM agents?"


# Step-Back (broader) question generate gareko
step_back_question = generate_queries_step_back.invoke({"question": question})


# Generate bhayeko Step-Back question print garne
print(step_back_question)

How do large language models (LLM) process complex tasks?


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# Local Ollama LLM use gareko
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Final answer generate garna prompt template create gareko
response_prompt_template = """
You are an expert in world knowledge.

I am going to ask you a question.

Your response should be comprehensive and should use the provided contexts whenever they are relevant. If any context is not relevant, ignore it.

Normal Context:
{normal_context}

Step-Back Context:
{step_back_context}

Original Question:
{question}

Answer:
"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)


# Step-Back RAG chain create gareko
chain = (
    {
        # Original question bata retrieve gareko context
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Step-Back question bata retrieve gareko context
        "step_back_context": generate_queries_step_back | retriever,
        # Original question prompt ma pass gareko
        "question": lambda x: x["question"],
    }
    # Prompt template ma values inject gareko
    | response_prompt
    # Local Ollama LLM bata final answer generate gareko
    | llm
    # Output lai plain text ma convert gareko
    | StrOutputParser()
)


# Final Step-Back RAG execute gareko
response = chain.invoke(
    {
        "question": question,
    }
)


# Final answer print garne
print(response)

According to the provided context, task decomposition is a key component of an LLM-powered autonomous agent system. In this context, it refers to the process by which the agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.

In the context of building agents with LLM as its core controller, task decomposition is crucial for several reasons:

1. **Efficient handling of complex tasks**: By breaking down large tasks into smaller subtasks, the agent can more effectively handle complex tasks that would be difficult or impossible to tackle in a single step.
2. **Improved planning and decision-making**: Task decomposition allows the agent to plan and make decisions at multiple levels, taking into account the relationships between different subtasks and their dependencies.
3. **Enhanced learning and adaptation**: By decomposing tasks into smaller parts, the agent can learn from its mistakes and refine its approach for future steps, lead